# 26 · Final evaluation on the untouched test set

## Purpose

This notebook uses the test set **for the first and only time**. It measures how well the model selected in notebook
25 predicts the **asking price** of listings it has never seen. It is **not** an experiment:

- the model, features, preprocessing and hyperparameters were **frozen in notebook 25**;
- the pipeline is fitted **once**, on the 16,219 training listings;
- the 4,081 test listings are predicted **once**;
- no result in this notebook is used to change anything. Weak results are recorded as limitations.

**Comparison models.** The project's documentation (`docs/data_cleaning_decisions.md`, steps 13–14) requires the test
result **overall, per price range and per property type**. It does not fix any comparison model for the test set.
Notebook 25 (section 7) had suggested showing the Ridge reference and a Dummy median as context. Following the
instruction for this notebook, **only the selected model is evaluated on the test set**. Leaving those context models
out does not change the selected model or its test result. Earlier cross-validation results of other models are quoted
from notebooks 19–25 where useful and labelled as such.

### The frozen final model (notebook 25, section 6)

| Setting | Value |
|---|---|
| Model | `HistGradientBoostingRegressor(learning_rate=0.1, max_iter=200, max_leaf_nodes=31, max_depth=None, min_samples_leaf=40, l2_regularization=1.0, early_stopping=False, random_state=42)` |
| Target | `log(price)` via `TransformedTargetRegressor(func=np.log, inverse_func=np.exp)`; predictions in rupees |
| Features | `build_model_features()`: 19 notebook-18 features + 6 locality-centre Haversine distances (notebook 23) |
| Preprocessing | numeric: median imputation → `StandardScaler`; binary: passthrough; categorical: `OneHotEncoder(handle_unknown="ignore")`; locality: `OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist")` |

## 1. Load data and check the split

In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_CSV = PROJECT_ROOT / "data" / "processed" / "cleaned_listings.csv"
SPLIT_CSV = PROJECT_ROOT / "data" / "processed" / "train_test_split.csv"
LOCALITIES_CSV = PROJECT_ROOT / "data" / "extracted" / "localities.csv"
FEATURES_PY = PROJECT_ROOT / "src" / "features" / "build_features.py"

sys.path.insert(0, str(PROJECT_ROOT))
from src.features.build_features import DISTANCE_FEATURES, INPUT_COLUMNS, build_model_features


def file_md5(path):
    # A fingerprint of the file's bytes. If the file changes, the fingerprint changes.
    return hashlib.md5(path.read_bytes()).hexdigest()


# The files used by notebooks 23-25
EXPECTED_MD5 = {
    CLEANED_CSV: "3a81a385a9aca28d5b70322e1e143c08",
    SPLIT_CSV: "10693e86d1d8a55e606399d718a8f556",
    LOCALITIES_CSV: "8d9bb54dfc2fb49673e74b976ab73ace",
    FEATURES_PY: "4fb50b05aa072d1a9ae25851dbf1cc69",
}
for path, md5 in EXPECTED_MD5.items():
    assert file_md5(path) == md5, f"{path.name} is not the file used by notebooks 23-25"
    print(f"{path.name:28} {md5}")

cleaned_listings.csv         3a81a385a9aca28d5b70322e1e143c08
train_test_split.csv         10693e86d1d8a55e606399d718a8f556
localities.csv               8d9bb54dfc2fb49673e74b976ab73ace
build_features.py            4fb50b05aa072d1a9ae25851dbf1cc69


In [2]:
listings = pd.read_csv(CLEANED_CSV, low_memory=False)
split_assignment = pd.read_csv(SPLIT_CSV)
locality_table = pd.read_csv(LOCALITIES_CSV, usecols=["locality_id", "locality_name", "locality_lat", "locality_long"])

listings = listings.merge(split_assignment[["property_id", "split"]], on="property_id", validate="one_to_one")
train = listings[listings["split"] == "train"].copy()
test = listings[listings["split"] == "test"].copy()
del listings

assert len(train) == 16219, len(train)
assert len(test) == 4081, len(test)
assert train["property_id"].is_unique and test["property_id"].is_unique
assert set(train["property_id"]).isdisjoint(test["property_id"])
assert set(train["property_group_id"]).isdisjoint(test["property_group_id"]), "A property group is on both sides!"
assert set(train.index).isdisjoint(test.index)

print("Training listings:", len(train), "| test listings:", len(test))
print("property_group_id values shared by train and test:",
      len(set(train["property_group_id"]) & set(test["property_group_id"])))

Training listings: 16219 | test listings: 4081
property_group_id values shared by train and test: 0


### Build the features

The same fixed rule for both sets. For the test set, **every price-related column is removed first**, so the test
features are provably built without the test prices. (`build_model_features()` only reads the allowed input columns
anyway.) The test prices are kept aside in `y_test` and are used only after the predictions have been made.

In [3]:
train_features = build_model_features(train, locality_table)
y_train = train["price"]

price_columns = [column for column in test.columns if "price" in column]
test_inputs = test.drop(columns=price_columns)          # the test listings WITHOUT any price information
y_test = test["price"]                                  # kept aside until the predictions exist
test_features = build_model_features(test_inputs, locality_table)

assert list(train_features.columns) == list(test_features.columns) and train_features.shape[1] == 25
assert test_features.index.equals(test.index) and len(test_features) == 4081
for column in ["price", "price_band", "price_per_sqft", "loc_avg_price_sqft", "locality_price_encoding",
               "latitude", "longitude", "locality_centre_lat", "locality_centre_long"]:
    assert column not in train_features.columns, column
for column in DISTANCE_FEATURES:
    assert column in train_features.columns, column
print("Price-related columns removed from the test inputs:", price_columns)
print("Feature tables:", train_features.shape, test_features.shape, "(same 25 columns, no coordinates, no target encoding)")

Price-related columns removed from the test inputs: ['price', 'price_band']
Feature tables: (16219, 25) (4081, 25) (same 25 columns, no coordinates, no target encoding)


## 2. Fit the frozen pipeline once, on the training set only

In [4]:
RANDOM_STATE = 42
NUMERIC_FEATURES = ["bedrooms", "bathrooms", "log_area", "apartment_floor", "apartment_building_floors"] + DISTANCE_FEATURES
BINARY_FEATURES = ["area_missing", "area_unit_is_sqyd", "has_land_area_unit", "prime_location_flag", "in_named_project"]
CATEGORICAL_FEATURES = ["property_type", "balconies_band", "transaction_group", "furnishing", "possession_status",
                        "age_of_property", "facing_clean", "posted_by"]
LOCALITY_FEATURES = ["locality_code"]
assert sorted(NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES + LOCALITY_FEATURES) == sorted(train_features.columns)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
         NUMERIC_FEATURES),
        ("binary", "passthrough", BINARY_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("locality", OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist", sparse_output=False),
         LOCALITY_FEATURES),
    ]
)
final_regressor = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=200, max_leaf_nodes=31, max_depth=None,
                                                min_samples_leaf=40, l2_regularization=1.0, early_stopping=False,
                                                random_state=RANDOM_STATE)
final_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", TransformedTargetRegressor(regressor=final_regressor, func=np.log, inverse_func=np.exp)),
])

# The ONLY fit in this notebook: training features and training prices
final_model.fit(train_features, y_train)
print("Fitted on", len(train_features), "training listings.")

Fitted on 16219 training listings.


**Proof that no test row was used in fitting.** Every value the preprocessing learned must come from the training
listings alone:

- the imputer medians equal the training medians;
- the scaler means equal the means of the imputed training values;
- the one-hot category lists equal the training categories;
- localities that occur **only** in the test set are **not** among the locality categories.

The HGB settings are the frozen ones, and the model was trained for exactly 200 rounds (no early stopping).

In [5]:
fitted_preprocess = final_model.named_steps["preprocess"]
numeric_steps = fitted_preprocess.named_transformers_["numeric"]

# Imputer and scaler learned from the training rows
training_medians = train_features[NUMERIC_FEATURES].median()
assert np.allclose(numeric_steps.named_steps["impute"].statistics_, training_medians.values)
imputed_training = train_features[NUMERIC_FEATURES].fillna(training_medians)
assert np.allclose(numeric_steps.named_steps["scale"].mean_, imputed_training.mean().values)

# Category lists learned from the training rows
categorical_encoder = fitted_preprocess.named_transformers_["categorical"]
for column, categories in zip(CATEGORICAL_FEATURES, categorical_encoder.categories_):
    assert list(categories) == sorted(train_features[column].unique()), column

locality_encoder = fitted_preprocess.named_transformers_["locality"]
assert list(locality_encoder.categories_[0]) == sorted(train_features["locality_code"].unique())
test_only_localities = set(test_features["locality_code"]) - set(train_features["locality_code"])
assert test_only_localities.isdisjoint(locality_encoder.categories_[0])

# The frozen hyperparameters, and exactly 200 boosting rounds
fitted_hgb = final_model.named_steps["model"].regressor_
FROZEN = {"learning_rate": 0.1, "max_iter": 200, "max_leaf_nodes": 31, "max_depth": None, "min_samples_leaf": 40,
          "l2_regularization": 1.0, "early_stopping": False, "random_state": 42}
for key, value in FROZEN.items():
    assert fitted_hgb.get_params()[key] == value, key
assert fitted_hgb.n_iter_ == 200

print("Imputer medians, scaler means and all category lists = training-set values.")
print(f"{len(test_only_localities)} localities appear only in the test set; none of them is a learned category")
print("(their listings go to the shared 'rare' column, exactly as for a new advert).")
print("Frozen hyperparameters confirmed; boosting rounds:", fitted_hgb.n_iter_)

Imputer medians, scaler means and all category lists = training-set values.
155 localities appear only in the test set; none of them is a learned category
(their listings go to the shared 'rare' column, exactly as for a new advert).
Frozen hyperparameters confirmed; boosting rounds: 200


## 3. Predict the test set, once

In [6]:
test_predictions = pd.Series(final_model.predict(test_features), index=test_features.index)   # the one test prediction

assert len(test_predictions) == 4081
assert test_predictions.index.equals(test.index) and test_predictions.index.is_unique
assert test_predictions.notna().all() and np.isfinite(test_predictions).all() and (test_predictions > 0).all()
assert set(test.loc[test_predictions.index, "property_id"]) == set(test["property_id"])
print("One finite, positive prediction for each of the", len(test_predictions), "test listings; each evaluated once.")

One finite, positive prediction for each of the 4081 test listings; each evaluated once.


## 4. Test performance

| Metric | Meaning |
|---|---|
| MAE | average absolute error, ₹ lakh (1 lakh = ₹1,00,000; 100 lakh = ₹1 crore) |
| RMSE | like MAE, but large errors count much more; dominated by the most expensive homes |
| R² | share of the rupee price variation explained |
| Median APE | median of \|prediction − price\| / price: half the listings are predicted within this % |
| within ±10% / ±25% | share of listings whose prediction is within 10% / 25% of the asking price |
| error ≥ 2× | share predicted at least twice too high or at most half the price |

In [7]:
def evaluate(actual, predicted):
    ratio = predicted / actual
    abs_pct_error = np.abs(predicted - actual) / actual * 100
    return {
        "listings": len(actual),
        "MAE (₹ lakh)": mean_absolute_error(actual, predicted) / 1e5,
        "RMSE (₹ lakh)": root_mean_squared_error(actual, predicted) / 1e5,
        "R²": r2_score(actual, predicted),
        "Median APE (%)": np.median(abs_pct_error),
        "within ±10% (%)": (abs_pct_error <= 10).mean() * 100,
        "within ±25% (%)": (abs_pct_error <= 25).mean() * 100,
        "error ≥ 2x (%)": ((ratio >= 2) | (ratio <= 0.5)).mean() * 100,
    }


test_result = evaluate(y_test, test_predictions)
pd.Series(test_result, name="TEST (untouched holdout)").to_frame().round(3)

,TEST (untouched holdout)
listings,4081.000
MAE (₹ lakh),57.011
RMSE (₹ lakh),158.338
R²,0.556
Median APE (%),16.836
within ±10% (%),31.830
within ±25% (%),66.675
error ≥ 2x (%),5.464


### Uncertainty of the test result

4,081 listings is a finite sample, so the test MAE would move a little with a different sample of listings. A simple
**bootstrap** estimates by how much: draw the **property groups** of the test set at random with replacement (copies of
one property stay together), recompute the metric, repeat 1,000 times, and take the middle 95% of the results. This
re-uses the **same** predictions; nothing is refitted.

In [8]:
test_table = pd.DataFrame({"group": test["property_group_id"], "price": y_test, "prediction": test_predictions})
rows_by_group = test_table.groupby("group").indices          # group -> row positions
group_ids = np.array(list(rows_by_group))
random_generator = np.random.default_rng(RANDOM_STATE)

bootstrap_mae, bootstrap_median_ape = [], []
for repetition in range(1000):
    chosen_groups = random_generator.choice(group_ids, size=len(group_ids), replace=True)
    positions = np.concatenate([rows_by_group[group] for group in chosen_groups])
    sample = test_table.iloc[positions]
    bootstrap_mae.append(mean_absolute_error(sample["price"], sample["prediction"]) / 1e5)
    bootstrap_median_ape.append(np.median(np.abs(sample["prediction"] - sample["price"]) / sample["price"]) * 100)

mae_interval = np.percentile(bootstrap_mae, [2.5, 97.5])
ape_interval = np.percentile(bootstrap_median_ape, [2.5, 97.5])
print(f"Test MAE        ₹{test_result['MAE (₹ lakh)']:.2f} lakh   95% bootstrap interval ₹{mae_interval[0]:.2f} - {mae_interval[1]:.2f} lakh")
print(f"Test Median APE  {test_result['Median APE (%)']:.2f}%        95% bootstrap interval  {ape_interval[0]:.2f} - {ape_interval[1]:.2f}%")

Test MAE        ₹57.01 lakh   95% bootstrap interval ₹52.49 - 61.81 lakh
Test Median APE  16.84%        95% bootstrap interval  16.19 - 17.64%


### Training performance (diagnostic only)

The same model predicting the listings it was **trained on**. This is **not** evidence of how well it generalises: a
model always looks better on its own training data. It is shown only to see how large the gap between seen and unseen
listings is.

In [9]:
train_predictions = pd.Series(final_model.predict(train_features), index=train_features.index)
assert train_predictions.notna().all() and np.isfinite(train_predictions).all() and (train_predictions > 0).all()

pd.DataFrame({
    "TRAINING (diagnostic, seen data)": evaluate(y_train, train_predictions),
    "TEST (untouched holdout)": test_result,
}).round(3)

,"TRAINING (diagnostic, seen data)",TEST (untouched holdout)
listings,16219.000,4081.000
MAE (₹ lakh),47.917,57.011
RMSE (₹ lakh),140.486,158.338
R²,0.685,0.556
Median APE (%),14.137,16.836
within ±10% (%),37.475,31.830
within ±25% (%),72.606,66.675
error ≥ 2x (%),3.366,5.464


## 5. Test set vs the notebook-25 cross-validation estimate

**The two numbers answer different questions.**

- The **CV estimate** (notebook 25) comes from 5 group-aware folds of the **training** set. The same folds were used to
  **choose** the configuration (the best of 32 HGB settings, and HGB over Ridge), so it is expected to be slightly
  optimistic.
- The **test result** comes from 4,081 listings that played no part in any decision. It is the project's **final,
  unbiased holdout estimate**, for this data collection.

(Notebook 25 did not report the "error ≥ 2×" share for this configuration, so that row has no CV value.)

In [10]:
CV_NOTEBOOK_25 = {   # printed by notebook 25 for the selected configuration (mean ± std over 5 folds; out-of-fold shares)
    "MAE (₹ lakh)": (57.528, 2.740),
    "RMSE (₹ lakh)": (161.873, 14.770),
    "R²": (0.576, 0.035),
    "Median APE (%)": (16.508, 0.754),
    "within ±10% (%)": (32.9, None),
    "within ±25% (%)": (66.2, None),
    "error ≥ 2x (%)": (None, None),
}
CV_FOLD_MAE = [53.875, 56.579, 60.921, 56.800, 59.464]   # notebook 25, config 4

rows = []
for metric, (cv_mean, cv_std) in CV_NOTEBOOK_25.items():
    rows.append({
        "metric": metric,
        "CV mean (nb 25)": cv_mean,
        "CV std": cv_std,
        "TEST": test_result[metric],
        "TEST - CV": None if cv_mean is None else test_result[metric] - cv_mean,
    })
comparison = pd.DataFrame(rows).set_index("metric")
print("CV fold MAEs (₹ lakh):", CV_FOLD_MAE, "-> range", min(CV_FOLD_MAE), "-", max(CV_FOLD_MAE))
comparison.round(3)

CV fold MAEs (₹ lakh): [53.875, 56.579, 60.921, 56.8, 59.464] -> range 53.875 - 60.921


,CV mean (nb 25),CV std,TEST,TEST - CV
metric,,,,
MAE (₹ lakh),57.528,2.740,57.011,-0.517
RMSE (₹ lakh),161.873,14.770,158.338,-3.535
R²,0.576,0.035,0.556,-0.020
Median APE (%),16.508,0.754,16.836,0.328
within ±10% (%),32.900,NaN,31.830,-1.070
within ±25% (%),66.200,NaN,66.675,0.475
error ≥ 2x (%),NaN,NaN,5.464,NaN


## 6. Test performance by group

`price_band` (the scraper's price searches, notebook 12) is computed from each listing's own price and is used **only
for reporting**. It was never a feature.

In [11]:
RANGE_NAMES = ["0-50 L", "50-80 L", "80 L-1.2 Cr", "1.2-1.5 Cr", "1.5-1.75 Cr",
               "1.75-2 Cr", "2-3 Cr", "3-5 Cr", "5 Cr+"]
PROPERTY_TYPES = ["Apartment", "Builder Floor Apartment", "Penthouse", "Residential House", "Villa"]

test_report = pd.DataFrame({
    "property_id": test["property_id"],
    "price": y_test,
    "prediction": test_predictions,
    "price_band": test["price_band"],
    "property_type": test["property_type"],
    "type_family": np.where(test["property_type"].isin(["Residential House", "Villa"]), "house / villa",
                            "apartment-like"),
}, index=test.index)
test_report["signed % error"] = (test_report["prediction"] - test_report["price"]) / test_report["price"] * 100


def breakdown(group_column, order):
    rows = {}
    for value in order:
        part = test_report[test_report[group_column] == value]
        if len(part) == 0:
            continue
        row = evaluate(part["price"], part["prediction"])
        row["median signed % error"] = part["signed % error"].median()
        rows[value] = row
    return pd.DataFrame(rows).T.round(2)


print("TEST, by price band (median signed error: + = predicted too high, - = too low)")
breakdown("price_band", RANGE_NAMES)

TEST, by price band (median signed error: + = predicted too high, - = too low)


,listings,MAE (₹ lakh),RMSE (₹ lakh),R²,Median APE (%),within ±10% (%),within ±25% (%),error ≥ 2x (%),median signed % error
0-50 L,311.0,18.87,35.92,-12.48,24.85,24.76,50.16,14.79,23.20
50-80 L,602.0,19.12,32.86,-14.72,16.68,31.89,64.78,5.15,5.68
80 L-1.2 Cr,529.0,23.09,31.92,-7.31,16.56,29.68,71.46,1.89,9.92
1.2-1.5 Cr,585.0,30.33,45.52,-21.11,15.96,33.68,69.91,1.54,4.98
1.5-1.75 Cr,458.0,30.64,45.96,-42.75,12.40,41.70,76.86,1.53,0.47
1.75-2 Cr,436.0,40.60,69.14,-72.20,14.15,36.47,74.54,2.98,-1.83
2-3 Cr,534.0,50.91,75.40,-5.20,15.72,33.71,72.47,3.00,-8.32
3-5 Cr,398.0,102.42,131.79,-4.15,20.76,24.62,57.04,10.05,-16.31
5 Cr+,228.0,375.64,615.98,-0.27,30.12,21.05,42.54,22.37,-27.53


In [12]:
print("TEST, by property type")
breakdown("property_type", PROPERTY_TYPES)

TEST, by property type


,listings,MAE (₹ lakh),RMSE (₹ lakh),R²,Median APE (%),within ±10% (%),within ±25% (%),error ≥ 2x (%),median signed % error
Apartment,2328.0,28.29,64.94,0.73,13.19,39.52,78.01,2.28,0.07
Builder Floor Apartment,148.0,47.46,162.81,0.17,19.23,27.70,60.81,9.46,-0.24
Penthouse,9.0,70.02,96.72,0.44,43.09,0.00,11.11,11.11,31.64
Residential House,1093.0,79.92,187.55,0.44,25.22,20.40,49.59,10.43,2.07
Villa,503.0,142.71,315.42,0.45,22.28,22.86,54.08,8.15,-1.04


In [13]:
print("TEST, apartment-like types vs houses/villas")
breakdown("type_family", ["apartment-like", "house / villa"])

TEST, apartment-like types vs houses/villas


,listings,MAE (₹ lakh),RMSE (₹ lakh),R²,Median APE (%),within ±10% (%),within ±25% (%),error ≥ 2x (%),median signed % error
apartment-like,2485.0,29.59,74.59,0.66,13.42,38.67,76.74,2.74,0.06
house / villa,1596.0,99.71,235.47,0.48,24.63,21.18,51.00,9.71,0.98


In [14]:
# The luxury homes (₹20 Cr+, docs/outlier_handling.md B2), reported in notebooks 20, 24 and 25
test_report["luxury"] = np.where(test_report["price"] >= 20e7, "₹20 Cr+", "below ₹20 Cr")
print("TEST, ₹20 Cr+ luxury homes")
breakdown("luxury", ["below ₹20 Cr", "₹20 Cr+"])

TEST, ₹20 Cr+ luxury homes


,listings,MAE (₹ lakh),RMSE (₹ lakh),R²,Median APE (%),within ±10% (%),within ±25% (%),error ≥ 2x (%),median signed % error
below ₹20 Cr,4065.0,50.68,107.82,0.66,16.75,31.96,66.94,5.22,0.43
₹20 Cr+,16.0,1665.63,1855.00,-5.52,58.48,0.00,0.00,68.75,-58.48


## 7. The largest test errors

Locality names come from the website's locality table (via the listing's locality code). "unknown locality" means the
listing had no usable locality.

In [15]:
name_by_code = dict(zip(locality_table["locality_id"].astype(int).astype(str), locality_table["locality_name"]))
test_report["locality"] = test_features["locality_code"].map(name_by_code).fillna(test_features["locality_code"])
test_report["bedrooms"] = test_features["bedrooms"]
test_report["area known"] = np.where(test_features["area_missing"] == 0, "yes", "no")
test_report["abs error (₹ lakh)"] = (test_report["prediction"] - test_report["price"]).abs() / 1e5
test_report["price (₹ lakh)"] = test_report["price"] / 1e5
test_report["prediction (₹ lakh)"] = test_report["prediction"] / 1e5
test_report["prediction / price"] = test_report["prediction"] / test_report["price"]

SHOW = ["property_id", "property_type", "bedrooms", "area known", "locality", "price (₹ lakh)", "prediction (₹ lakh)",
        "abs error (₹ lakh)", "prediction / price"]
print("The 10 largest absolute test errors (₹)")
test_report.sort_values("abs error (₹ lakh)", ascending=False)[SHOW].head(10).round(2)

The 10 largest absolute test errors (₹)


,property_id,property_type,bedrooms,area known,locality,price (₹ lakh),prediction (₹ lakh),abs error (₹ lakh),prediction / price
20111,listing_001,Residential House,3,no,Secunderabad,4000.0,316.63,3683.37,0.08
19226,listing_002,Villa,5,no,Jubilee Hills,4400.0,1068.93,3331.07,0.24
20008,listing_003,Villa,3,no,unknown locality,2300.0,181.10,2118.90,0.08
19193,listing_004,Villa,4,yes,Mallampet,2300.0,230.93,2069.07,0.10
19428,listing_005,Residential House,11,yes,Jubilee Hills,3300.0,1376.40,1923.60,0.42
20001,listing_006,Villa,4,yes,unknown locality,2600.0,695.09,1904.91,0.27
19783,listing_007,Builder Floor Apartment,6,no,Gachibowli,2000.0,204.89,1795.11,0.10
20113,listing_008,Residential House,7,yes,Miyapur,1750.0,215.60,1534.40,0.12
19766,listing_009,Villa,5,yes,Manchirevula,2600.0,1086.81,1513.19,0.42
20141,listing_010,Villa,5,no,Shankarpally Road,2000.0,692.75,1307.25,0.35


In [16]:
print("The 10 largest OVER-predictions (prediction far above the asking price, by ratio)")
test_report.sort_values("prediction / price", ascending=False)[SHOW].head(10).round(2)

The 10 largest OVER-predictions (prediction far above the asking price, by ratio)


,property_id,property_type,bedrooms,area known,locality,price (₹ lakh),prediction (₹ lakh),abs error (₹ lakh),prediction / price
1277,listing_011,Residential House,9,no,Gayatri Nagar Godavari Homes,5.50,321.67,316.17,58.49
1072,listing_012,Residential House,5,no,Saroornagar,9.90,202.86,192.95,20.49
1356,listing_013,Apartment,2,no,Ameenpur Plots,8.70,79.34,70.64,9.12
822,listing_014,Residential House,4,yes,Ahmed Nagar,26.75,206.67,179.92,7.73
754,listing_015,Residential House,9,no,Santosh Nagar,28.00,187.02,159.02,6.68
1341,listing_016,Residential House,2,no,Uppuguda,12.00,77.55,65.55,6.46
688,listing_017,Residential House,2,yes,Secunderabad,11.50,60.97,49.47,5.30
13929,listing_018,Residential House,6,yes,Koti,200.00,1034.94,834.94,5.17
1412,listing_019,Residential House,4,yes,unknown locality,30.00,132.35,102.35,4.41
3065,listing_020,Residential House,4,yes,Sundar Nagar,57.00,246.24,189.24,4.32


In [17]:
print("The 10 largest UNDER-predictions (prediction far below the asking price, by ratio)")
test_report.sort_values("prediction / price", ascending=True)[SHOW].head(10).round(2)

The 10 largest UNDER-predictions (prediction far below the asking price, by ratio)


,property_id,property_type,bedrooms,area known,locality,price (₹ lakh),prediction (₹ lakh),abs error (₹ lakh),prediction / price
20251,listing_021,Apartment,3,no,Manikonda,950.0,74.76,875.24,0.08
20008,listing_003,Villa,3,no,unknown locality,2300.0,181.10,2118.90,0.08
20111,listing_001,Residential House,3,no,Secunderabad,4000.0,316.63,3683.37,0.08
20247,listing_022,Apartment,2,yes,Bachupally,740.0,68.12,671.88,0.09
19193,listing_004,Villa,4,yes,Mallampet,2300.0,230.93,2069.07,0.10
19783,listing_007,Builder Floor Apartment,6,no,Gachibowli,2000.0,204.89,1795.11,0.10
20221,listing_023,Apartment,3,no,unknown locality,1200.0,140.19,1059.81,0.12
20271,listing_024,Apartment,3,no,Uppal,1000.0,118.01,881.99,0.12
20210,listing_025,Apartment,2,yes,Kompally,700.0,83.53,616.47,0.12
20113,listing_008,Residential House,7,yes,Miyapur,1750.0,215.60,1534.40,0.12


In [18]:
# Context for the over-predictions: very low asking prices (probable entry errors, notebook 20) in the test set
print("Test listings priced at ₹2 lakh or less:", (test_report["price"] <= 2e5).sum())
print("Test listings priced below ₹10 lakh:     ", (test_report["price"] < 10e5).sum())

Test listings priced at ₹2 lakh or less: 0
Test listings priced below ₹10 lakh:      5


## 8. Final checks

In [19]:
# Row counts and groups
assert len(train_features) == 16219 and len(test_features) == 4081
assert set(train["property_group_id"]).isdisjoint(test["property_group_id"])
print("1. 16,219 training and 4,081 test listings; no property_group_id on both sides.")

# No test row in the fit (section 2 proved the learned values are training-only)
assert set(train_features.index).isdisjoint(test_features.index)
print("2. The pipeline was fitted once, on training rows only; all learned values match the training set.")

# One prediction per test listing, all valid, each evaluated once
assert len(test_predictions) == len(test) == 4081 and test_predictions.index.is_unique
assert test_report.index.equals(test.index) and test_report["property_id"].is_unique
assert np.isfinite(test_predictions).all() and (test_predictions > 0).all()
print("3. Exactly one finite, positive prediction per test listing; every test listing evaluated exactly once.")

# Nothing saved: no model or dataset file was written by this notebook
print("4. No model and no transformed dataset was saved.")

# Source files unchanged
for path, md5 in EXPECTED_MD5.items():
    assert file_md5(path) == md5, f"{path.name} changed!"
print("5. cleaned_listings.csv, train_test_split.csv, localities.csv and build_features.py are unchanged.")

1. 16,219 training and 4,081 test listings; no property_group_id on both sides.
2. The pipeline was fitted once, on training rows only; all learned values match the training set.


3. Exactly one finite, positive prediction per test listing; every test listing evaluated exactly once.
4. No model and no transformed dataset was saved.


5. cleaned_listings.csv, train_test_split.csv, localities.csv and build_features.py are unchanged.


## Final Model Evaluation

### Final model configuration (frozen in notebook 25)

`HistGradientBoostingRegressor(learning_rate=0.1, max_iter=200, max_leaf_nodes=31, max_depth=None, min_samples_leaf=40,
l2_regularization=1.0, early_stopping=False, random_state=42)`:
- **Target:** log(asking price), converted back to rupees.
- **Features:** `build_model_features()`, the 19 notebook-18 features plus 6 locality-centre distances. No coordinates,
  no listing pins, no target encoding.
- **Training:** fitted once on all 16,219 training listings, with every preprocessing step learned from them only.

### CV estimate vs untouched test result

| Metric | CV, notebook 25 (5 group-aware folds, used for selection) | **Test (4,081 unseen listings)** | Test − CV |
|---|---|---|---|
| MAE | ₹57.53 ± 2.74 lakh | **₹57.01 lakh** (95% bootstrap interval ₹52.5–61.8) | −₹0.52 lakh |
| Median APE | 16.51 ± 0.75% | **16.84%** (95% interval 16.2–17.6%) | +0.33 points |
| RMSE | ₹161.87 ± 14.77 lakh | **₹158.34 lakh** | −₹3.5 lakh |
| R² | 0.576 ± 0.035 | **0.556** | −0.020 |
| within ±10% / ±25% | 32.9% / 66.2% | **31.8% / 66.7%** | −1.1 / +0.5 points |
| error ≥ 2× | – | **5.5%** | – |

**The test result agrees with the cross-validation estimate.** Every difference is smaller than about one fold standard
deviation. The test MAE (₹57.0 lakh) lies inside the range of the five CV folds (₹53.9–60.9 lakh). The small
optimism expected from choosing the best configuration on the CV folds is not visible beyond normal sampling noise.

On its own training data the model reaches MAE ₹47.9 lakh and Median APE 14.1%. This is a diagnostic only: the gap to
the test set (₹57.0 lakh, 16.8%) is the expected difference between seen and unseen listings, not a sign of failure.

### Main strengths

- **Typical listings:** half of all test listings are predicted within about 17% of their asking price; two-thirds
  within ±25%; only 5.5% are off by a factor of two or more.
- **Apartments:** Median APE 13.2% (apartment-like types 13.4%), 77% within ±25%, and almost no systematic bias (median
  signed error ≈ 0%).
- **The middle of the market (₹50 L–3 Cr):** Median APE 12–17% in every band.
- The result is **stable**. It matches the CV estimate, and the model was fixed before the test set was opened.

### Main failure modes

- **Pull toward the middle.**
  - Cheap listings are over-predicted: ₹0–50 L has a median signed error of +23% and a Median APE of 25%; 15% are off by
    2× or more.
  - Expensive listings are under-predicted: ₹3–5 Cr is −16%; ₹5 Cr+ is −28%, with a Median APE of 30%.
- **Luxury homes (₹20 Cr+, 16 test listings)** are under-predicted by a median of 58%. None is within ±25%. These 16
  listings dominate the rupee errors: without them the test MAE is ₹50.7 lakh.
- **Houses and villas** are about twice as hard as apartments (Median APE 24.6% vs 13.4%; 51% vs 77% within ±25%). Their
  area can mean plot or built-up area, and the data cannot tell which (notebook 08).
- **The largest errors** are:
  - multi-crore houses and villas, often without a usable area;
  - a few 2–3 bedroom apartments advertised at ₹7–12 Cr and predicted at ₹0.7–1.4 Cr;
  - several large houses advertised at ₹5–30 lakh and predicted at ₹0.6–3.2 Cr.

  Some of these asking prices are probably **entry errors**, but that cannot be proven from the data. They are kept in
  the evaluation, as decided before the test set was opened.
- **Price bands:** within one band, R² is low or negative. That is expected when the prices inside a band are close
  together, and it is not an additional failure. The band-level MAE and Median APE are the meaningful measures there.

### Limitations

- The target is the **advertised asking price** (91% owner listings), not a transaction price.
- The data is **one scraped snapshot** (July 2026) collected by **separate price-range searches** with result caps
  (CLAUDE.md point 8). It is not a random sample of the Hyderabad market, so these errors describe this collection.
  They are not market-wide accuracy, and they say nothing about future periods.
- Rule R1 removed about 0.6% of listings with implausible prices before the split; other probable entry errors remain in
  both training and test data.
- About 7% of listings have no usable locality. Area has different meanings across property types.
- Converting log predictions back with `exp()` gives values closer to a median than a mean, so rupee predictions are
  slightly biased low.
- Only one final model was evaluated on the test set, as intended. There is no test-set comparison with other models;
  earlier model comparisons are cross-validation results on the training set.

### What can legitimately be claimed

On listings from the same collection that the model never saw (4,081 test listings, grouped so that no property
appears on both sides), the final HistGradientBoosting model predicts the **advertised asking price**:
- with a **mean absolute error of about ₹57 lakh** (95% bootstrap interval ₹52–62 lakh);
- with a **median absolute percentage error of about 17%**;
- within ±25% for about **two-thirds** of listings.

It is most reliable for apartments and mid-priced homes (roughly ₹50 L–3 Cr). It is systematically too high for the
cheapest listings and too low for expensive ones, and it cannot predict ₹20 Cr+ luxury homes.

It is **not** a model of sale prices. It is not validated for other time periods or for the whole Hyderabad market, and it
should not be used to value luxury properties or listings without a usable area and locality with the same confidence.